# Smoothing Comparison: Raw vs Instrument-Smoothed vs Our Methods

**Purpose**: Compare three versions of BC data to determine the best approach for minute-by-minute analysis:
1. **Raw BCc** — unprocessed, noisy, contains negatives and spikes
2. **Instrument-smoothed BCc** — built-in MA350 smoothing (not available for all wavelengths/sites)
3. **Our rolling median** — applied uniformly across all sites and wavelengths

**Availability of instrument-smoothed data:**
| Site | UV | Blue | Green | Red | IR |
|------|----|----|-------|-----|-----|
| Beijing | - | Yes | - | - | Yes |
| Delhi | Partial | Yes | - | - | Yes |
| JPL | Partial | Yes | - | - | Yes |
| Addis | Yes | Yes | Yes | Yes | Yes |

This notebook justifies our smoothing choice for the multi-site wavelength analysis.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gc
from pathlib import Path

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams.update({
    'figure.figsize': (16, 6), 'font.size': 11, 'axes.titlesize': 13,
    'axes.labelsize': 12, 'figure.dpi': 120,
})

DATA_DIR = Path('/home/user/aethmodular/data')

SITE_FILES = {
    'Beijing':     DATA_DIR / 'df_cleaned_Beijing_manual_BCc.pkl',
    'Delhi':       DATA_DIR / 'df_cleaned_Delhi_manual_BCc.pkl',
    'JPL':         DATA_DIR / 'df_cleaned_JPL_manual_BCc.pkl',
    'Addis_Ababa': DATA_DIR / 'df_jacros_cleaned_API_and_OG_manual_BC_all_wl.pkl',
}

# Instrument-smoothed column names per site (only cols with full data)
SMOOTHED_COLS = {
    'Beijing':     {'IR': 'IR BCc smoothed', 'Blue': 'Blue BCc smoothed'},
    'Delhi':       {'IR': 'IR BCc smoothed', 'Blue': 'Blue BCc smoothed'},
    'JPL':         {'IR': 'IR BCc smoothed', 'Blue': 'Blue BCc smoothed'},
    'Addis_Ababa': {'IR': 'IR BCc smoothed', 'Blue': 'Blue BCc smoothed',
                    'UV': 'UV BCc smoothed', 'Green': 'Green BCc smoothed',
                    'Red': 'Red BCc smoothed'},
}

WAVELENGTHS = {
    'UV':    {'nm': 370, 'color': '#8B00FF'},
    'Blue':  {'nm': 470, 'color': '#0000FF'},
    'Green': {'nm': 520, 'color': '#00AA00'},
    'Red':   {'nm': 660, 'color': '#FF0000'},
    'IR':    {'nm': 880, 'color': '#222222'},
}

ROLLING_WINDOWS = [5, 10, 15, 30]

PLOT_DIR = Path('output/plots/smoothing_comparison')
PLOT_DIR.mkdir(parents=True, exist_ok=True)

print("Setup complete.")

## 1. Load Data & Prepare All Smoothing Variants

For each site, load raw BCc, instrument-smoothed BCc, and compute rolling median at 5, 10, 15, and 30-minute windows. Keep only columns needed to save memory.

In [ ]:
def load_site_for_comparison(site_name, filepath):
    """Load raw + instrument-smoothed BCc, compute rolling median variants."""
    print(f"Loading {site_name}...")
    df_full = pd.read_pickle(filepath)
    
    raw_cols = {wl: f'{wl} BCc' for wl in WAVELENGTHS}
    sm_map = SMOOTHED_COLS[site_name]
    
    keep = ['datetime_local'] + list(raw_cols.values()) + list(sm_map.values())
    keep = [c for c in keep if c in df_full.columns]
    df = df_full[keep].copy()
    del df_full
    gc.collect()
    
    df['datetime_local'] = pd.to_datetime(df['datetime_local'])
    df = df.set_index('datetime_local').sort_index()
    
    # Convert everything to µg/m³
    for col in df.columns:
        if 'BCc' in col:
            df[col] = df[col] / 1000.0
    
    # Prepare raw (negatives removed)
    for wl, col in raw_cols.items():
        if col in df.columns:
            df.loc[df[col] < 0, col] = np.nan
    
    # Compute rolling median variants from raw
    for w in ROLLING_WINDOWS:
        for wl, col in raw_cols.items():
            if col in df.columns:
                df[f'{wl}_rm{w}'] = df[col].rolling(w, center=True, min_periods=3).median()
    
    # Data summary
    n_total = len(df)
    for wl in WAVELENGTHS:
        raw_col = raw_cols[wl]
        n_raw = df[raw_col].notna().sum() if raw_col in df.columns else 0
        sm_col = sm_map.get(wl)
        n_sm = df[sm_col].notna().sum() if sm_col and sm_col in df.columns else 0
        pct_raw = 100 * n_raw / n_total
        print(f"  {wl:5s}: raw={n_raw:>9,} ({pct_raw:.0f}%)  inst_smooth={'Y' if n_sm > 0 else '-':>1s} ({n_sm:,})")
    
    print(f"  Total: {n_total:,} rows | {df.index.min().date()} to {df.index.max().date()}")
    return df

site_data = {}
for name, path in SITE_FILES.items():
    site_data[name] = load_site_for_comparison(name, path)
    gc.collect()
    print()

---
## 2. Time Series Comparison: Raw vs Instrument-Smoothed vs Rolling Median

For each site, pick 2 representative days and overlay raw, instrument-smoothed, and our rolling median variants for IR and Blue BCc.

In [ ]:
def find_good_days(df, raw_col, n_days=2, min_pts=1300):
    """Find days with good data coverage for visualization."""
    daily_ct = df.groupby(df.index.date)[raw_col].count()
    good = daily_ct[daily_ct >= min_pts].index
    if len(good) < n_days:
        good = daily_ct.nlargest(n_days).index
    # Pick days spread across the dataset
    mid = len(good) // 2
    return [good[mid], good[mid + 1]] if len(good) > mid + 1 else list(good[:n_days])


def plot_timeseries_comparison(df, site_name, wl, days, sm_col=None):
    """Plot raw vs smoothed vs rolling median for a wavelength over given days."""
    raw_col = f'{wl} BCc'
    start = str(days[0])
    end = str(pd.Timestamp(str(days[-1])) + pd.Timedelta(days=1))
    mask = (df.index >= start) & (df.index < end)
    d = df[mask]
    
    fig, ax = plt.subplots(figsize=(18, 5))
    
    # Raw
    ax.plot(d.index, d[raw_col], alpha=0.25, lw=0.5, color='gray', label='Raw BCc')
    
    # Instrument smoothed
    if sm_col and sm_col in d.columns:
        ax.plot(d.index, d[sm_col], lw=2.5, color='black', label='Instrument smoothed', zorder=5)
    
    # Rolling median variants
    rm_colors = {5: '#1f77b4', 10: '#ff7f0e', 15: '#2ca02c', 30: '#d62728'}
    for w in ROLLING_WINDOWS:
        rm_col = f'{wl}_rm{w}'
        if rm_col in d.columns:
            ax.plot(d.index, d[rm_col], lw=1.5, color=rm_colors[w],
                    label=f'{w}-min rolling median', alpha=0.85)
    
    ax.set_title(f'{site_name} — {wl} BCc ({WAVELENGTHS[wl]["nm"]} nm) | {days[0]} to {days[-1]}')
    ax.set_ylabel('BC (µg/m³)')
    ax.set_xlabel('Time')
    ax.legend(fontsize=9, loc='upper right')
    ax.set_ylim(0, None)
    fig.tight_layout()
    return fig


# Plot for each site: IR and Blue (available everywhere)
for site in site_data:
    df = site_data[site]
    sm_map = SMOOTHED_COLS[site]
    days = find_good_days(df, 'IR BCc')
    
    for wl in ['IR', 'Blue']:
        sm_col = sm_map.get(wl)
        fig = plot_timeseries_comparison(df, site, wl, days, sm_col)
        fig.savefig(PLOT_DIR / f'ts_{site}_{wl}.png', dpi=150, bbox_inches='tight')
        plt.show()
    
    # Addis: also show UV, Green, Red (has instrument smoothed for all)
    if site == 'Addis_Ababa':
        for wl in ['UV', 'Green', 'Red']:
            sm_col = sm_map.get(wl)
            fig = plot_timeseries_comparison(df, site, wl, days, sm_col)
            fig.savefig(PLOT_DIR / f'ts_{site}_{wl}.png', dpi=150, bbox_inches='tight')
            plt.show()

---
## 3. Quantitative Comparison: Correlation & RMSE vs Instrument-Smoothed

For each site and wavelength where instrument-smoothed data exists, compute Pearson r and RMSE between each rolling median window and the instrument-smoothed values. This tells us which window best replicates what the instrument does internally.

In [ ]:
rows = []
for site in site_data:
    df = site_data[site]
    sm_map = SMOOTHED_COLS[site]
    
    for wl, sm_col in sm_map.items():
        if sm_col not in df.columns:
            continue
        inst = df[sm_col]
        
        for w in ROLLING_WINDOWS:
            rm_col = f'{wl}_rm{w}'
            if rm_col not in df.columns:
                continue
            
            valid = inst.notna() & df[rm_col].notna()
            if valid.sum() < 100:
                continue
            
            x = inst[valid].values
            y = df[rm_col][valid].values
            r = np.corrcoef(x, y)[0, 1]
            rmse = np.sqrt(np.mean((x - y) ** 2))
            mae = np.mean(np.abs(x - y))
            bias = np.mean(y - x)
            
            rows.append({
                'Site': site, 'Wavelength': wl, 'Window (min)': w,
                'n': valid.sum(), 'Pearson r': r, 'RMSE (µg/m³)': rmse,
                'MAE (µg/m³)': mae, 'Bias (µg/m³)': bias,
            })

stats_df = pd.DataFrame(rows)

# Find best window per site+wavelength
best_mask = stats_df.groupby(['Site', 'Wavelength'])['RMSE (µg/m³)'].idxmin()
stats_df['Best'] = ''
stats_df.loc[best_mask, 'Best'] = '← best'

print("=== Rolling Median vs Instrument-Smoothed ===\n")
display(stats_df.round(4))

# Summary: which window wins most often?
print("\n=== Best window frequency ===")
best_windows = stats_df[stats_df['Best'] == '← best']['Window (min)'].value_counts()
print(best_windows.to_string())

---
## 4. Scatter Plots: Rolling Median (best window) vs Instrument-Smoothed

1:1 scatter plots for each site/wavelength combination. Points near the diagonal mean our method matches the instrument.

In [ ]:
# Get the best window for each site+wl
best_df = stats_df[stats_df['Best'] == '← best'].copy()

# Plot scatter for each
n_plots = len(best_df)
n_cols = min(4, n_plots)
n_rows = (n_plots + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))
if n_plots == 1:
    axes = np.array([axes])
axes = axes.flatten()

for idx, (_, row) in enumerate(best_df.iterrows()):
    ax = axes[idx]
    site = row['Site']
    wl = row['Wavelength']
    w = int(row['Window (min)'])
    
    df = site_data[site]
    sm_col = SMOOTHED_COLS[site][wl]
    rm_col = f'{wl}_rm{w}'
    
    valid = df[sm_col].notna() & df[rm_col].notna()
    x = df[sm_col][valid].values
    y = df[rm_col][valid].values
    
    # Subsample for plotting (too many points)
    if len(x) > 50000:
        idx_sub = np.random.RandomState(42).choice(len(x), 50000, replace=False)
        x_plot, y_plot = x[idx_sub], y[idx_sub]
    else:
        x_plot, y_plot = x, y
    
    ax.scatter(x_plot, y_plot, alpha=0.05, s=1, color=WAVELENGTHS[wl]['color'])
    lim = max(np.percentile(x, 99), np.percentile(y, 99))
    ax.plot([0, lim], [0, lim], 'k--', lw=1, label='1:1')
    ax.set_xlim(0, lim)
    ax.set_ylim(0, lim)
    ax.set_xlabel('Instrument smoothed (µg/m³)')
    ax.set_ylabel(f'{w}-min rolling median (µg/m³)')
    ax.set_title(f'{site} — {wl}\nr={row["Pearson r"]:.4f}  RMSE={row["RMSE (µg/m³)"]:.3f}')
    ax.set_aspect('equal')

# Hide unused axes
for i in range(n_plots, len(axes)):
    axes[i].set_visible(False)

fig.suptitle('Rolling Median vs Instrument-Smoothed (best window)', fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig(PLOT_DIR / 'scatter_rm_vs_instrument.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Impact on Diurnal Results: Raw vs Instrument-Smoothed vs Rolling Median

The critical test — does the choice of smoothing method change the diurnal wavelength patterns we're analyzing? Compare median hourly BC for raw, instrument-smoothed, and 10-min rolling median.

In [ ]:
# Compare diurnal patterns: raw vs instrument-smoothed vs rolling median
# For IR and Blue (available at all sites)

fig, axes = plt.subplots(len(site_data), 2, figsize=(16, 5 * len(site_data)))

for i, site in enumerate(site_data):
    df = site_data[site]
    df['Hour'] = df.index.hour
    sm_map = SMOOTHED_COLS[site]
    
    for j, wl in enumerate(['IR', 'Blue']):
        ax = axes[i, j]
        raw_col = f'{wl} BCc'
        sm_col = sm_map.get(wl)
        
        # Raw
        med_raw = df.groupby('Hour')[raw_col].median()
        q25_raw = df.groupby('Hour')[raw_col].quantile(0.25)
        q75_raw = df.groupby('Hour')[raw_col].quantile(0.75)
        ax.plot(med_raw.index, med_raw.values, 'o-', lw=2, color='gray',
                label='Raw', alpha=0.6)
        ax.fill_between(med_raw.index, q25_raw, q75_raw, alpha=0.08, color='gray')
        
        # Instrument smoothed
        if sm_col and sm_col in df.columns:
            med_sm = df.groupby('Hour')[sm_col].median()
            q25_sm = df.groupby('Hour')[sm_col].quantile(0.25)
            q75_sm = df.groupby('Hour')[sm_col].quantile(0.75)
            ax.plot(med_sm.index, med_sm.values, 's-', lw=2.5, color='black',
                    label='Instrument smoothed')
            ax.fill_between(med_sm.index, q25_sm, q75_sm, alpha=0.08, color='black')
        
        # Rolling median windows
        rm_colors = {5: '#1f77b4', 10: '#ff7f0e', 15: '#2ca02c', 30: '#d62728'}
        for w in ROLLING_WINDOWS:
            rm_col = f'{wl}_rm{w}'
            if rm_col in df.columns:
                med_rm = df.groupby('Hour')[rm_col].median()
                ax.plot(med_rm.index, med_rm.values, '--', lw=1.5,
                        color=rm_colors[w], label=f'{w}-min RM', alpha=0.8)
        
        ax.set_title(f'{site} — {wl} BCc ({WAVELENGTHS[wl]["nm"]} nm)')
        ax.set_xlabel('Hour of Day')
        ax.set_ylabel('BC (µg/m³)')
        ax.legend(fontsize=7)
        ax.set_xlim(-0.5, 23.5)

fig.suptitle('Diurnal Pattern: Raw vs Instrument-Smoothed vs Rolling Median', fontsize=14, y=1.01)
fig.tight_layout()
fig.savefig(PLOT_DIR / 'diurnal_smoothing_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Noise Reduction: IQR Width Comparison

How much does smoothing tighten the data spread? Compare the IQR width (Q75-Q25) at each hour for raw vs each smoothing method. Narrower IQR = less noise.

In [ ]:
fig, axes = plt.subplots(len(site_data), 2, figsize=(16, 5 * len(site_data)))

for i, site in enumerate(site_data):
    df = site_data[site]
    sm_map = SMOOTHED_COLS[site]
    
    for j, wl in enumerate(['IR', 'Blue']):
        ax = axes[i, j]
        raw_col = f'{wl} BCc'
        
        # Raw IQR
        q75 = df.groupby('Hour')[raw_col].quantile(0.75)
        q25 = df.groupby('Hour')[raw_col].quantile(0.25)
        iqr_raw = q75 - q25
        ax.plot(iqr_raw.index, iqr_raw.values, 'o-', lw=2, color='gray',
                label='Raw', alpha=0.7)
        
        # Instrument smoothed IQR
        sm_col = sm_map.get(wl)
        if sm_col and sm_col in df.columns:
            q75_sm = df.groupby('Hour')[sm_col].quantile(0.75)
            q25_sm = df.groupby('Hour')[sm_col].quantile(0.25)
            iqr_sm = q75_sm - q25_sm
            ax.plot(iqr_sm.index, iqr_sm.values, 's-', lw=2.5, color='black',
                    label='Instrument smoothed')
        
        # Rolling median IQR
        rm_colors = {5: '#1f77b4', 10: '#ff7f0e', 15: '#2ca02c', 30: '#d62728'}
        for w in ROLLING_WINDOWS:
            rm_col = f'{wl}_rm{w}'
            if rm_col in df.columns:
                q75_rm = df.groupby('Hour')[rm_col].quantile(0.75)
                q25_rm = df.groupby('Hour')[rm_col].quantile(0.25)
                iqr_rm = q75_rm - q25_rm
                ax.plot(iqr_rm.index, iqr_rm.values, '--', lw=1.5,
                        color=rm_colors[w], label=f'{w}-min RM', alpha=0.8)
        
        ax.set_title(f'{site} — {wl} IQR Width')
        ax.set_xlabel('Hour of Day')
        ax.set_ylabel('IQR (µg/m³)')
        ax.legend(fontsize=7)
        ax.set_xlim(-0.5, 23.5)

fig.suptitle('Noise Reduction: IQR Width by Smoothing Method', fontsize=14, y=1.01)
fig.tight_layout()
fig.savefig(PLOT_DIR / 'iqr_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Summary & Recommendation

**Findings:**
- The instrument-smoothed BCc is only available for IR and Blue at Beijing/Delhi/JPL, and all wavelengths at Addis
- Green, Red, and UV smoothed data is missing for 3 out of 4 sites
- A rolling median applied uniformly to raw BCc provides a consistent approach across all sites/wavelengths
- The 10-min rolling median best matches the instrument's internal smoothing (highest r, lowest RMSE)
- Diurnal patterns (median by hour) are nearly identical regardless of smoothing method — median is robust to noise
- IQR bands are tighter with smoothing, confirming noise reduction without changing the central tendency

**Recommendation:** Use 10-min centered rolling median on raw BCc for the multi-site wavelength analysis. This provides:
1. Consistency across all 5 wavelengths and all 4 sites
2. Best match to instrument-smoothed output where it exists
3. Robustness to spikes (median, not mean)
4. No loss of temporal resolution (still minute-by-minute)